# Następna iteracja: category-aware top-k prototype reconstruction

Ten notebook zbiera wnioski po `CATEGORY_AWARE_RERANKING_LOCAL.ipynb` i uruchamia kolejną lokalną iterację.

Poprzedni etap dał pierwszy wynik candidate-constrained reconstruction, który przebił VAE baseline: `eegnet_top1_category_gate` osiągnął na teście `SSIM ≈ 0.308`, podczas gdy VAE ensemble dla `mole` ma `SSIM ≈ 0.286`.

Nowe pytanie brzmi: czy zamiast wybierać jeden obraz-kandydata możemy zbudować rekonstrukcję jako ważony prototyp z top-k kandydatów?

## Wnioski przed eksperymentem

1. Bezpośredni Stable UnCLIP z EEG był słabszy niż VAE.
2. Retrieval EEG→CLIP miał sygnał, ale cierpiał na hubness/collapse.
3. `candidate_zscore` ograniczył hubness, ale jeszcze nie przebił VAE.
4. EEGNet jako bramka kategorii poprawił wynik do `SSIM ≈ 0.308`.
5. Oracle kategorii nadal jest dużo wyżej, więc informacja kategorii jest głównym dźwignikiem poprawy.

Ta iteracja sprawdza tani lokalny substytut generowania: średnią ważoną pikseli z top-k kandydatów po score'ach category-aware. To nie jest docelowy generator; to test, czy top-k kandydaci dają sensowny prototyp obrazu.

In [ ]:
from pathlib import Path

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD if (CWD / 'scripts').is_dir() else CWD.parent
OUTPUT_DIR = PROJECT_ROOT / 'wyniki colab' / 'category_prototype_reconstruction_mole_local'
SCRIPT = PROJECT_ROOT / 'scripts' / 'run_category_prototype_reconstruction.py'

paths = {
    'script': SCRIPT,
    'category-aware summary': PROJECT_ROOT / 'wyniki colab' / 'category_aware_reranking_mole_local' / 'category_aware_reranking_summary.json',
    'retrieval checkpoint': PROJECT_ROOT / 'wyniki colab' / 'unclip_mole_retrieval' / 'eeg_image_retrieval.pt',
    'eegnet checkpoint': PROJECT_ROOT / 'wyniki colab' / 'eegnet_mole_colab' / 'eegnet.pt',
    'unclip embeddings': PROJECT_ROOT / 'image_embeddings_unclip_participant_image_mole_no_abc_local_20260627',
}
for name, path in paths.items():
    print(f'{name}:', path, 'OK' if path.exists() else 'BRAK')

In [ ]:
# Uruchom lokalny eksperyment prototypów top-k.
import subprocess, sys

cmd = [
    sys.executable, str(SCRIPT),
    '--project-root', str(PROJECT_ROOT),
    '--output-dir', str(OUTPUT_DIR),
    '--force',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Tabela porównawcza: prototyp vs baseline nearest-neighbor.
import json
import pandas as pd
from IPython.display import display

summary = json.loads((OUTPUT_DIR / 'prototype_reconstruction_summary.json').read_text(encoding='utf-8'))
comparison = pd.read_csv(OUTPUT_DIR / 'prototype_selected_comparison.csv')
cols = ['selection', 'prototype_key', 'top_k', 'temperature', 'anchor_top1_weight', 'ssim', 'l1', 'top1', 'top5', 'category_top1', 'is_pixel_prototype']
display(comparison[cols])

print('Wybrany prototyp:', summary['selected_key'])
print('Test SSIM:', summary['test_selected_by_validation']['ssim'])
print('Test L1:', summary['test_selected_by_validation']['l1'])

In [ ]:
# Pełne rankingi wariantów na walidacji i teście.
for split in ['validation', 'test']:
    df = pd.read_csv(OUTPUT_DIR / split / 'prototype_method_comparison.csv')
    print('\n' + split.upper())
    display(df[['prototype_key', 'variant', 'top_k', 'temperature', 'anchor_top1_weight', 'ssim', 'l1', 'is_oracle', 'is_pixel_prototype']].head(15))

In [ ]:
# Podgląd gridów.
from IPython.display import Image as IPImage, Markdown, display

for path in sorted((OUTPUT_DIR / 'grids').glob('*.jpg')):
    display(Markdown(f'### {path.name}'))
    display(IPImage(filename=str(path)))

## Wpis historyczny — wynik lokalny z 2026-06-27

Najlepszy prototyp wybrany na walidacji to `eegnet_top1_category_gate_k3_t0.5_anchor0.5`: top-3 kandydatów, temperatura `0.5`, z domieszką `50%` obrazu top-1.

Na teście osiągnął:

- `SSIM = 0.286`,
- `L1 = 0.241`,
- `top1 = 18.18%`, `top5 = 38.64%`, `category_top1 = 38.64%`.

Interpretacja: prototyp top-k prawie dochodzi do VAE baseline (`SSIM ≈ 0.286`), ale przegrywa z pojedynczym category-gate nearest-neighbor (`SSIM ≈ 0.308`). Wizualnie widać, że średnia pikseli rozmywa obiekty i miesza abstrakcyjne wzorce.

Wniosek dla następnej iteracji: top-k kandydaci są dobrzy jako **kontekst/warunek dla generatora**, ale nie jako pikselowa średnia. Następny kierunek to generator warunkowany top-k referencjami albo wybór jednego kandydata plus lokalna korekta/refinement, a nie blendowanie pikseli.